# 0. Problem
## 1164. Product Price at a Given Date — Medium
All products start at price 10. For each product, return the price on `2019-08-16`: the latest change on/before that date, or 10 if no such change exists.

Official: https://leetcode.com/problems/product-price-at-a-given-date/

# 1. Setup

In [ ]:
import pandas as pd
products_rows=[(1,20,"2019-08-14"),(2,50,"2019-08-14"),(1,30,"2019-08-15"),(1,35,"2019-08-16"),(2,65,"2019-08-17"),(3,20,"2019-08-18")]
products_pd=pd.DataFrame(products_rows,columns=["product_id","new_price","change_date"])
products_pd["change_date"]=pd.to_datetime(products_pd["change_date"])
products_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark=SparkSession.builder.getOrCreate()
products_spark=spark.createDataFrame(products_rows,["product_id","new_price","change_date"]).withColumn("change_date",F.to_date("change_date"))
products_spark.createOrReplaceTempView("Products")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""
WITH ranked AS (
    SELECT product_id,new_price,change_date,
           ROW_NUMBER() OVER(PARTITION BY product_id ORDER BY change_date DESC) AS rn
    FROM Products
    WHERE change_date<=DATE('2019-08-16')
), all_products AS (
    SELECT DISTINCT product_id FROM Products
)
SELECT p.product_id, COALESCE(r.new_price,10) AS price
FROM all_products p
LEFT JOIN ranked r
  ON p.product_id=r.product_id AND r.rn=1
ORDER BY p.product_id
""")
sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
target=pd.Timestamp("2019-08-16")
all_products=products_pd[["product_id"]].drop_duplicates()
eligible=products_pd.loc[products_pd["change_date"]<=target].sort_values(["product_id","change_date"])
latest=eligible.groupby("product_id",as_index=False).tail(1)[["product_id","new_price"]]
result_pd=(all_products.merge(latest,on="product_id",how="left").assign(price=lambda d:d["new_price"].fillna(10).astype(int)).drop(columns="new_price").sort_values("product_id").reset_index(drop=True))
result_pd

# 4. PySpark Solution

In [ ]:
all_products=products_spark.select("product_id").distinct()
w=Window.partitionBy("product_id").orderBy(F.col("change_date").desc())
latest=(products_spark.filter(F.col("change_date")<=F.to_date(F.lit("2019-08-16"))).withColumn("rn",F.row_number().over(w)).filter(F.col("rn")==1).select("product_id","new_price"))
result_spark=(all_products.join(latest,on="product_id",how="left").select("product_id",F.coalesce(F.col("new_price"),F.lit(10)).alias("price")).orderBy("product_id"))
result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| latest row before cutoff | `ROW_NUMBER() ... DESC` | sort + `.groupby().tail(1)` | Window `row_number()` |
| preserve products with no eligible row | all-products + `LEFT JOIN` | left `.merge()` | left `.join()` |
| default | `COALESCE(...,10)` | `.fillna(10)` | `F.coalesce(...,F.lit(10))` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Products

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: products_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: products_spark